In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import linear_model
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd
import matplotlib
import seaborn as sns
import sklearn
import imblearn
import matplotlib.pyplot as plt
import time
import sklearn.metrics as m
import xgboost as xgb
import warnings
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import log_loss,cohen_kappa_score
from imblearn.metrics import geometric_mean_score
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
df = pd.read_csv("/content/drive/MyDrive/cicddos2019_dataset.csv")
df = df.drop(['Unnamed: 0','Class'], axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86275 entries, 0 to 86274
Data columns (total 56 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Bwd Packets/s             86275 non-null  float64
 1   Packet Length Min         86275 non-null  float64
 2   Packet Length Max         86275 non-null  float64
 3   Packet Length Mean        86275 non-null  float64
 4   Packet Length Std         86275 non-null  float64
 5   Packet Length Variance    86275 non-null  float64
 6   RST Flag Count            86275 non-null  int64  
 7   URG Flag Count            86275 non-null  int64  
 8   CWE Flag Count            86275 non-null  int64  
 9   Down/Up Ratio             86275 non-null  float64
 10  Avg Packet Size           86275 non-null  float64
 11  Avg Fwd Segment Size      86275 non-null  float64
 12  Avg Bwd Segment Size      86275 non-null  float64
 13  Subflow Fwd Packets       86275 non-null  int64  
 14  Subflo

In [ ]:
df['Label'].unique()


array(['DrDoS_NTP', 'Syn', 'TFTP', 'UDP', 'Benign', 'UDP-lag',
       'DrDoS_UDP', 'MSSQL', 'DrDoS_MSSQL', 'Portmap', 'LDAP',
       'DrDoS_SNMP', 'DrDoS_DNS', 'DrDoS_LDAP', 'NetBIOS',
       'DrDoS_NetBIOS', 'UDPLag', 'WebDDoS'], dtype=object)

In [4]:
attack_mapping = {
    'DrDoS_NTP': 'DrDoS',
    'Benign': 'Normal',
    'DrDoS_UDP': 'DrDoS',
    'DrDoS_MSSQL': 'DrDoS',
    'DrDoS_SNMP': 'DrDoS',
    'DrDoS_DNS': 'DrDoS',
    'DrDoS_LDAP': 'DrDoS',
    'DrDoS_NetBIOS': 'DrDoS',
    'UDP-lag': 'UDP',
    'UDPLag':'UDP',
    'WebDDoS':'DrDoS'

}

# Map sub-types to main types and create a new column
df['Label'] = df['Label'].replace(attack_mapping)

In [5]:
df['Label'].value_counts()

,count
Label,
DrDoS,29295
TFTP,19869
Normal,19619
Syn,9847
UDP,5308
MSSQL,1726
LDAP,375
Portmap,124
NetBIOS,112


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.ticker as tkr
def func(x, pos):  # formatter function takes tick label and tick position
    s = '%d' % x
    groups = []
    while s and s[-1].isdigit():
        groups.append(s[-3:])
        s = s[:-3]
    return s + ','.join(reversed(groups))

y_format = tkr.FuncFormatter(func)  # make formatter
plt.figure(figsize=(8,5), dpi = 200)
p=sns.countplot(x='Label',data= df , palette='bright')
p.yaxis.set_major_formatter(y_format)
p.tick_params(labelsize=13)
plt.xticks(rotation = 85)
plt.xlabel("attack",fontsize=12)
plt.ylabel("Count",fontsize=12)
plt.show()

In [6]:
y = df['Label']
x = df.drop(['Label'],axis=1)

from sklearn import preprocessing
le = preprocessing.LabelEncoder()
for column in x.columns:
    if x[column].dtype == 'object':
        x[column] = le.fit_transform(x[column])

In [7]:
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.2, shuffle=True)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy='auto', k_neighbors=3, random_state=0)

# Resample only the training data using SMOTE (after train/test split)
X_train, Y_train = smote.fit_resample(X_train, Y_train)

dfb = pd.DataFrame(X_train)
dfb['Label'] = Y_train

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.ticker as tkr
def func(x, pos):  # formatter function takes tick label and tick position
    s = '%d' % x
    groups = []
    while s and s[-1].isdigit():
        groups.append(s[-3:])
        s = s[:-3]
    return s + ','.join(reversed(groups))

y_format = tkr.FuncFormatter(func)  # make formatter
plt.figure(figsize=(8,5), dpi = 200)
p=sns.countplot(x='Label',data= dfb, palette='bright' )
p.yaxis.set_major_formatter(y_format)
p.tick_params(labelsize=13)
plt.xticks(rotation = 85)
plt.xlabel("attack",fontsize=12)
plt.ylabel("Count",fontsize=12)
plt.show()

In [ ]:
x.head()

,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,RST Flag Count,URG Flag Count,CWE Flag Count,Down/Up Ratio,Avg Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init Fwd Win Bytes,Init Bwd Win Bytes,Fwd Act Data Packets,Fwd Seg Size Min,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s
0,0.000000,296.0,440.0,437.09090,20.362576,414.63452,0,0,0,0.0,441.551030,437.06122,0.0,98,42832,0,0,-1,-1,97,20,17,449,98,0,42832.0,0.0,440.0,296.0,437.06122,20.465117,0.0,0.0,0.0,0.0,9.539421e+07,218262.810000,4.628866e+00,1.249276e+01,44.0,0.0,449.0,4.628866e+00,1.249276e+01,44.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1960,0,218262.810000
1,0.359366,6.0,6.0,6.00000,0.000000,0.00000,0,0,0,0.0,6.428571,6.00000,6.0,8,48,6,36,5840,0,7,20,6,16696066,8,6,48.0,36.0,6.0,6.0,6.00000,0.000000,6.0,6.0,6.0,0.0,5.031125e+00,0.838521,1.284313e+06,2.963577e+06,8999759.0,0.0,16696066.0,2.385152e+06,3.808248e+06,8999862.0,0.0,7696201.0,1539240.2,2915302.0,6696477.0,1.0,0,160,120,0.479155
2,0.000000,516.0,516.0,516.00000,0.000000,0.00000,0,0,0,0.0,645.000000,516.00000,0.0,4,2064,0,0,-1,-1,3,-1,17,3008437,4,0,2064.0,0.0,516.0,516.0,516.00000,0.000000,0.0,0.0,0.0,0.0,6.860706e+02,1.329594,1.002812e+06,1.736921e+06,3008436.0,0.0,3008437.0,1.002812e+06,1.736921e+06,3008436.0,0.0,0.0,0.0,0.0,0.0,0.0,0,-4,0,1.329594
3,0.000000,516.0,516.0,516.00000,0.000000,0.00000,0,0,0,0.0,645.000000,516.00000,0.0,4,2064,0,0,-1,-1,3,20,17,3112318,4,0,2064.0,0.0,516.0,516.0,516.00000,0.000000,0.0,0.0,0.0,0.0,6.631713e+02,1.285216,1.037439e+06,1.796896e+06,3112316.0,1.0,3112318.0,1.037439e+06,1.796896e+06,3112316.0,1.0,0.0,0.0,0.0,0.0,0.0,0,80,0,1.285216
4,0.000000,321.0,393.0,344.14285,33.617596,1130.14280,0,0,0,0.0,401.500000,348.00000,0.0,6,2088,0,0,-1,-1,5,20,17,215106,6,0,2088.0,0.0,393.0,321.0,348.00000,35.088460,0.0,0.0,0.0,0.0,9.706842e+03,27.893225,4.302120e+04,5.890395e+04,109536.0,0.0,215106.0,4.302120e+04,5.890395e+04,109536.0,0.0,0.0,0.0,0.0,0.0,0.0,0,120,0,27.893225


# results with original featues

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(random_state=10, solver='lbfgs',max_iter=50,multi_class='auto',C=1.0)
start = time.time()
clf.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=clf.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
# kf = KFold(n_splits=5, shuffle=True)
# clf = LogisticRegression(random_state=10, solver='lbfgs',max_iter=5,multi_class='auto',C=1.0)
# score = cross_val_score(clf, X, y, cv= kf, scoring="accuracy")
# print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

Training time: 27.642394542694092s
accuracy score 0.22275321916898977
error rate: 0.7772467808310102
              precision    recall  f1-score   support

       DrDoS       0.45      0.00      0.01      5903
        LDAP       0.21      0.97      0.34      5755
       MSSQL       0.20      0.09      0.13      5968
     NetBIOS       0.05      0.01      0.01      5794
      Normal       0.49      0.17      0.25      5822
     Portmap       0.00      0.00      0.00      5865
         Syn       0.24      0.72      0.36      5901
        TFTP       0.45      0.05      0.10      5833
         UDP       0.01      0.00      0.01      5890

    accuracy                           0.22     52731
   macro avg       0.23      0.22      0.13     52731
weighted avg       0.23      0.22      0.13     52731



array([[  27, 5294,   42,   13,    2,    0,  398,   23,  104],
       [  14, 5561,  180,    0,    0,    0,    0,    0,    0],
       [   0, 5383,  554,    0,    0,    0,   22,    5,    4],
       [   0, 5292,  239,   41,    0,    0,  112,  103,    7],
       [   0,  344,  453,  582,  983,    0, 1831,    3, 1626],
       [   0, 4436,  428,    2,  252,    0,  622,  120,    5],
       [   0,    4,  898,   13,  740,    0, 4246,    0,    0],
       [  10,   21,    0,   11,    0,    0, 5480,  310,    1],
       [   9,  385,    6,  172,   18,    0, 5151,  125,   24]])

Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier
print("RF")
clf = RandomForestClassifier(n_estimators=2,max_depth=2, random_state=100,criterion='entropy')
start = time.time()
clf.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=clf.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
# kf = KFold(n_splits=10, shuffle=True)
# clf = RandomForestClassifier(n_estimators=20,max_depth=5, random_state=100,criterion='entropy')
# score = cross_val_score(clf, x, y, cv= kf, scoring="accuracy")
# print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

RF
Training time: 3.4320321083068848s
accuracy score 0.5125448028673835
error rate: 0.48745519713261654
              precision    recall  f1-score   support

       DrDoS       0.95      0.78      0.86      5903
        LDAP       0.44      0.98      0.61      5755
       MSSQL       0.00      0.00      0.00      5968
     NetBIOS       0.43      0.98      0.59      5794
      Normal       0.00      0.00      0.00      5822
     Portmap       0.00      0.00      0.00      5865
         Syn       0.00      0.00      0.00      5901
        TFTP       0.95      0.99      0.97      5833
         UDP       0.34      0.90      0.49      5890

    accuracy                           0.51     52731
   macro avg       0.34      0.51      0.39     52731
weighted avg       0.34      0.51      0.39     52731



array([[4619,  675,    0,  183,    0,    0,    0,   24,  402],
       [ 129, 5626,    0,    0,    0,    0,    0,    0,    0],
       [  39, 5903,    0,    0,    0,    0,    0,   26,    0],
       [   0,   30,    0, 5702,    0,    0,    0,    0,   62],
       [  18,   22,    0, 1378,    0,    0,    0,  230, 4174],
       [   0,  317,    0, 4873,    0,    0,    0,    0,  675],
       [   0,    4,    0,  914,    0,    0,    0,    0, 4983],
       [   2,   18,    0,   12,    0,    0,    0, 5791,   10],
       [  70,  182,    0,  339,    0,    0,    0,   10, 5289]])

Decesion Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
DT = DecisionTreeClassifier(max_depth=2, random_state=100,criterion='entropy')
start = time.time()
DT.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=DT.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
# kf = KFold(n_splits=10, shuffle=True)
# DT = DecisionTreeClassifier(max_depth=2, random_state=100,criterion='entropy')
# score = cross_val_score(DT, x, y, cv= kf, scoring="accuracy")
# print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

Training time: 9.253193855285645s
accuracy score 0.43862244220667157
error rate: 0.5613775577933284
              precision    recall  f1-score   support

       DrDoS       0.00      0.00      0.00      5903
        LDAP       0.45      0.99      0.62      5755
       MSSQL       0.00      0.00      0.00      5968
     NetBIOS       0.38      1.00      0.55      5794
      Normal       0.00      0.00      0.00      5822
     Portmap       0.00      0.00      0.00      5865
         Syn       0.41      1.00      0.58      5901
        TFTP       0.54      0.99      0.70      5833
         UDP       0.00      0.00      0.00      5890

    accuracy                           0.44     52731
   macro avg       0.20      0.44      0.27     52731
weighted avg       0.20      0.44      0.27     52731



array([[   0,  530,    0,  731,    0,    0,   34, 4608,    0],
       [   0, 5669,    0,    0,    0,    0,    0,   86,    0],
       [   0, 5927,    0,    0,    0,    0,    0,   41,    0],
       [   0,    0,    0, 5771,    0,    0,   23,    0,    0],
       [   0,   18,    0,   50,    0,    0, 5679,   75,    0],
       [   0,  332,    0, 4307,    0,    0, 1221,    5,    0],
       [   0,    4,    0,    0,    0,    0, 5897,    0,    0],
       [   0,   19,    0,    1,    0,    0,   21, 5792,    0],
       [   0,  157,    0, 4236,    0,    0, 1436,   61,    0]])

knn

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
k = 3  # You can choose a different value for k
knn_classifier = KNeighborsClassifier(n_neighbors=k)
start = time.time()
knn_classifier.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=knn_classifier.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
# kf = KFold(n_splits=10, shuffle=True)
# knn_classifier = KNeighborsClassifier(n_neighbors=k)
# score = cross_val_score(knn_classifier, x, y, cv= kf, scoring="accuracy")
# print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

Training time: 0.793743371963501s
accuracy score 0.9330564563539474
error rate: 0.0669435436460526
              precision    recall  f1-score   support

       DrDoS       0.90      0.83      0.87      5903
        LDAP       0.96      0.96      0.96      5755
       MSSQL       0.91      0.95      0.93      5968
     NetBIOS       0.91      0.90      0.90      5794
      Normal       0.99      0.98      0.99      5822
     Portmap       0.91      0.90      0.90      5865
         Syn       0.96      0.95      0.95      5901
        TFTP       1.00      0.99      0.99      5833
         UDP       0.88      0.93      0.90      5890

    accuracy                           0.93     52731
   macro avg       0.93      0.93      0.93     52731
weighted avg       0.93      0.93      0.93     52731



array([[4928,  148,  318,   28,   17,   37,    3,    4,  420],
       [ 140, 5503,   95,   12,    0,    3,    0,    2,    0],
       [ 144,   76, 5673,   12,    0,   22,   11,    1,   29],
       [  63,   10,   21, 5230,    2,  429,   17,    0,   22],
       [  24,    0,    9,    5, 5730,   17,   26,    4,    7],
       [  39,    0,   63,  468,    4, 5263,   10,    0,   18],
       [   4,    0,    9,    2,   17,   20, 5623,    7,  219],
       [  11,    0,    7,    0,    5,    2,   11, 5785,   12],
       [ 128,    3,   55,   17,   13,   22,  183,    3, 5466]])

Gradient boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb_model = GradientBoostingClassifier(n_estimators=2, learning_rate=0.1, max_depth=3, random_state=42)
start = time.time()
gb_model.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=gb_model.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
# kf = KFold(n_splits=10, shuffle=True)
# gb_model = GradientBoostingClassifier(n_estimators=2, learning_rate=0.1, max_depth=3, random_state=42)
# score = cross_val_score(gb_model, x, y, cv= kf, scoring="accuracy")
# print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

Training time: 38.50871419906616s
accuracy score 0.8849822684948133
error rate: 0.11501773150518668
              precision    recall  f1-score   support

       DrDoS       0.93      0.83      0.88      5903
        LDAP       0.92      0.99      0.95      5755
       MSSQL       0.88      0.94      0.91      5968
     NetBIOS       0.65      0.95      0.77      5794
      Normal       0.92      0.97      0.95      5822
     Portmap       0.85      0.39      0.53      5865
         Syn       1.00      0.99      0.99      5901
        TFTP       1.00      0.99      1.00      5833
         UDP       0.92      0.92      0.92      5890

    accuracy                           0.88     52731
   macro avg       0.90      0.89      0.88     52731
weighted avg       0.90      0.88      0.88     52731



array([[4879,  252,  283,   47,   23,   14,    0,    0,  405],
       [   0, 5681,   74,    0,    0,    0,    0,    0,    0],
       [  67,  268, 5628,    0,    0,    0,    0,    5,    0],
       [   1,    0,    0, 5529,    0,  264,    0,    0,    0],
       [   0,    0,    5,   86, 5640,   62,    1,    0,   28],
       [  42,    0,  267, 2872,  400, 2274,    4,    0,    6],
       [   0,    0,    4,    0,   20,   21, 5816,    0,   40],
       [   9,    0,   24,    2,    4,    2,    0, 5780,   12],
       [ 243,    1,  110,   32,   23,   42,    0,    0, 5439]])

MLP Classifier

In [ ]:
from sklearn.neural_network import MLPClassifier
# Create an MLP model
mlp_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42)
start = time.time()
mlp_model.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=mlp_model.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
# kf = KFold(n_splits=10, shuffle=True)
#mlp_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42)
# score = cross_val_score(mlp_model, x, y, cv= kf, scoring="accuracy")
# print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

Training time: 165.96178889274597s
accuracy score 0.24873414120725948
error rate: 0.7512658587927405
              precision    recall  f1-score   support

       DrDoS       0.97      0.24      0.39      5903
        LDAP       0.00      0.00      0.00      5755
       MSSQL       0.00      0.00      0.00      5968
     NetBIOS       0.66      0.01      0.02      5794
      Normal       0.99      0.70      0.82      5822
     Portmap       0.99      0.11      0.20      5865
         Syn       0.98      0.15      0.26      5901
        TFTP       0.13      1.00      0.23      5833
         UDP       0.82      0.03      0.06      5890

    accuracy                           0.25     52731
   macro avg       0.62      0.25      0.22     52731
weighted avg       0.62      0.25      0.22     52731



array([[1431,    2,    0,    0,   11,    0,    0, 4445,   14],
       [   0,    0,    0,    0,    0,    0,    0, 5755,    0],
       [   0,    0,    0,    0,    0,    0,    0, 5968,    0],
       [   5,    0,    0,   48,    7,    0,    0, 5734,    0],
       [  18,    0,    0,   25, 4099,    0,   14, 1660,    6],
       [   0,    0,    0,    0,    0,  635,    0, 5230,    0],
       [   0,    0,    0,    0,    3,    9,  889, 4987,   13],
       [   0,    0,    0,    0,    1,    0,    0, 5822,   10],
       [  19,    0,    0,    0,    4,    0,    0, 5675,  192]])

# results with proposed featues

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
print("KNC")
RF=KNeighborsClassifier(n_neighbors=3)
rfPre1=RF.fit(x, y).predict_proba(x)
rf=pd.DataFrame(rfPre1)

KNC


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=4, learning_rate=0.1, max_depth=4, random_state=0)
gbPre1=gb.fit(x, y).predict_proba(x)
gbn=pd.DataFrame(gbPre1)

In [ ]:
result = pd.concat([gbn,rf], axis=1).reindex(gbn.index)
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(result,y,test_size = 0.2,random_state=0,shuffle=True)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
print("RF")
clf = RandomForestClassifier(n_estimators=20,max_depth=20, random_state=0)
start = time.time()
clf.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=clf.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
kf = KFold(n_splits=10, shuffle=True)
clf = RandomForestClassifier(n_estimators=20,max_depth=20, random_state=0)
score = cross_val_score(clf, result, y, cv= kf, scoring="accuracy")
print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

RF
Training time: 5.508709907531738s
accuracy score 0.9771292029356545
error rate: 0.02287079706434547
              precision    recall  f1-score   support

       DrDoS       0.96      0.93      0.95      5873
        LDAP       0.97      0.99      0.98      5905
       MSSQL       0.96      0.98      0.97      5848
     NetBIOS       0.96      0.97      0.97      5900
      Normal       1.00      1.00      1.00      5855
     Portmap       0.96      0.96      0.96      5803
         Syn       1.00      1.00      1.00      5871
        TFTP       1.00      1.00      1.00      5870
         UDP       0.98      0.97      0.97      5806

    accuracy                           0.98     52731
   macro avg       0.98      0.98      0.98     52731
weighted avg       0.98      0.98      0.98     52731

0.9776 accuracy with a standard deviation of 0.0010


array([[5445,  136,  183,   13,    6,   12,    1,    0,   77],
       [  11, 5869,   25,    0,    0,    0,    0,    0,    0],
       [  36,   23, 5758,    0,    1,   13,    0,    0,   17],
       [   3,    0,    0, 5729,    0,  167,    0,    0,    1],
       [   7,    0,    0,    1, 5840,    1,    1,    4,    1],
       [   3,    0,    6,  228,    0, 5559,    3,    0,    4],
       [   0,    0,    1,    0,    0,    8, 5852,    0,   10],
       [   3,    0,    1,    0,    1,    0,    2, 5856,    7],
       [ 141,    0,   30,    0,    1,   10,    5,    2, 5617]])

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(random_state=10, solver='lbfgs',max_iter=50,multi_class='auto',C=1.0)
start = time.time()
clf.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=clf.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
kf = KFold(n_splits=10, shuffle=True)
clf = LogisticRegression(random_state=10, solver='lbfgs',max_iter=5,multi_class='auto',C=1.0)
score = cross_val_score(clf, result, y, cv= kf, scoring="accuracy")
print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

Training time: 6.491850852966309s
accuracy score 0.9688987502607574
error rate: 0.031101249739242554
              precision    recall  f1-score   support

       DrDoS       0.97      0.88      0.92      5873
        LDAP       0.96      0.99      0.97      5905
       MSSQL       0.95      0.97      0.96      5848
     NetBIOS       0.95      0.97      0.96      5900
      Normal       1.00      1.00      1.00      5855
     Portmap       0.95      0.94      0.95      5803
         Syn       1.00      0.99      1.00      5871
        TFTP       1.00      1.00      1.00      5870
         UDP       0.94      0.98      0.96      5806

    accuracy                           0.97     52731
   macro avg       0.97      0.97      0.97     52731
weighted avg       0.97      0.97      0.97     52731

0.9607 accuracy with a standard deviation of 0.0012


array([[5157,  214,  168,   20,   12,   17,    2,    1,  282],
       [   4, 5865,   36,    0,    0,    0,    0,    0,    0],
       [  82,   49, 5681,    3,    1,   12,    0,    3,   17],
       [   1,    0,    0, 5697,    0,  202,    0,    0,    0],
       [   8,    0,    3,    1, 5831,    4,    1,    4,    3],
       [   9,    0,   34,  270,    0, 5478,    4,    0,    8],
       [   1,    0,    1,    0,    2,   13, 5837,    0,   17],
       [   0,    0,    6,    0,    1,    0,    3, 5853,    7],
       [  63,    0,   31,    2,    1,   15,    0,    2, 5692]])

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
k = 3  # You can choose a different value for k
RF = KNeighborsClassifier(n_neighbors=k)
start = time.time()
RF.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=RF.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
kf = KFold(n_splits=10, shuffle=True)
RF = KNeighborsClassifier(n_neighbors=k)
score = cross_val_score(RF, result, y, cv= kf, scoring="accuracy")
print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

Training time: 1.1669440269470215s
accuracy score 0.9743225047884546
error rate: 0.025677495211545387
              precision    recall  f1-score   support

       DrDoS       0.96      0.92      0.94      5873
        LDAP       0.97      0.99      0.98      5905
       MSSQL       0.96      0.98      0.97      5848
     NetBIOS       0.95      0.97      0.96      5900
      Normal       1.00      1.00      1.00      5855
     Portmap       0.96      0.95      0.95      5803
         Syn       0.99      1.00      1.00      5871
        TFTP       1.00      1.00      1.00      5870
         UDP       0.98      0.97      0.97      5806

    accuracy                           0.97     52731
   macro avg       0.97      0.97      0.97     52731
weighted avg       0.97      0.97      0.97     52731

0.9197 accuracy with a standard deviation of 0.0049


array([[5432,  144,  185,   15,    5,   11,    1,    1,   79],
       [  12, 5863,   29,    0,    0,    0,    0,    0,    1],
       [  73,   21, 5703,    0,    1,   12,    2,    0,   36],
       [   4,    0,    0, 5718,    0,  177,    0,    0,    1],
       [  10,    0,    0,    2, 5837,    1,    2,    2,    1],
       [   8,    0,   12,  271,    1, 5494,   11,    1,    5],
       [   1,    0,    0,    0,    2,    3, 5855,    2,    8],
       [   1,    0,    0,    0,    2,    0,    2, 5862,    3],
       [ 140,    0,   17,    1,    2,   13,   15,    5, 5613]])

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb_model = GradientBoostingClassifier(n_estimators=5, learning_rate=0.1, max_depth=5, random_state=0)
start = time.time()
gb_model.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=gb_model.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
kf = KFold(n_splits=10, shuffle=True)
gb_model = GradientBoostingClassifier(n_estimators=5, learning_rate=0.1, max_depth=5, random_state=0)
score = cross_val_score(gb_model, result, y, cv= kf, scoring="accuracy")
print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

Training time: 24.039998769760132s
accuracy score 0.974417325671806
error rate: 0.025582674328194055
              precision    recall  f1-score   support

       DrDoS       0.96      0.92      0.94      5873
        LDAP       0.97      1.00      0.98      5905
       MSSQL       0.96      0.98      0.97      5848
     NetBIOS       0.95      0.97      0.96      5900
      Normal       1.00      1.00      1.00      5855
     Portmap       0.96      0.95      0.95      5803
         Syn       1.00      1.00      1.00      5871
        TFTP       1.00      1.00      1.00      5870
         UDP       0.98      0.96      0.97      5806

    accuracy                           0.97     52731
   macro avg       0.97      0.97      0.97     52731
weighted avg       0.97      0.97      0.97     52731

0.9745 accuracy with a standard deviation of 0.0006


array([[5407,  158,  221,   19,   11,   15,    3,    0,   39],
       [   4, 5889,   12,    0,    0,    0,    0,    0,    0],
       [  30,   42, 5740,    0,    1,   17,    0,    0,   18],
       [   1,    0,    0, 5722,    0,  177,    0,    0,    0],
       [   5,    0,    0,    1, 5841,    4,    1,    3,    0],
       [   3,    0,    6,  267,    0, 5511,    3,    0,   13],
       [   1,    0,    1,    0,    1,   10, 5842,    1,   15],
       [   0,    0,    1,    0,    1,    0,    2, 5859,    7],
       [ 194,    0,   29,    1,    2,    8,    0,    1, 5571]])

In [ ]:
from sklearn.tree import DecisionTreeClassifier
DT = DecisionTreeClassifier(max_depth=6, random_state=100,criterion='entropy')
start = time.time()
DT.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=DT.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
kf = KFold(n_splits=10, shuffle=True)
DT = DecisionTreeClassifier(max_depth=6, random_state=100,criterion='entropy')
score = cross_val_score(DT, result, y, cv= kf, scoring="accuracy")
print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

Training time: 2.705451250076294s
accuracy score 0.9582029546187252
error rate: 0.04179704538127482
              precision    recall  f1-score   support

       DrDoS       0.93      0.92      0.93      5873
        LDAP       0.97      0.97      0.97      5905
       MSSQL       0.92      0.99      0.96      5848
     NetBIOS       0.95      0.90      0.93      5900
      Normal       0.92      1.00      0.96      5855
     Portmap       0.96      0.93      0.95      5803
         Syn       1.00      0.99      0.99      5871
        TFTP       1.00      1.00      1.00      5870
         UDP       0.98      0.94      0.96      5806

    accuracy                           0.96     52731
   macro avg       0.96      0.96      0.96     52731
weighted avg       0.96      0.96      0.96     52731

0.9581 accuracy with a standard deviation of 0.0012


array([[5422,  157,  260,   14,    8,    7,    0,    0,    5],
       [  42, 5700,  120,    0,   42,    0,    0,    0,    1],
       [  11,   30, 5806,    0,    0,    0,    0,    1,    0],
       [  12,    0,    3, 5308,  349,  198,    0,    0,   30],
       [  14,    0,    4,    0, 5827,    4,    2,    0,    4],
       [  29,    0,   47,  238,   46, 5406,    2,    0,   35],
       [   3,    1,    1,    0,   50,   13, 5784,    0,   19],
       [   3,    0,    0,    0,   13,    0,    0, 5844,   10],
       [ 300,    0,   68,    0,    1,    7,    0,    0, 5430]])

In [ ]:
from sklearn.neural_network import MLPClassifier
# Create an MLP model
mlp_model = MLPClassifier(hidden_layer_sizes=(10,), max_iter=10, random_state=42)
start = time.time()
mlp_model.fit(X_train, Y_train)
stop = time.time()
print(f"Training time: {stop - start}s")
dtP3=mlp_model.predict(X_test)
print('accuracy score',accuracy_score(Y_test,dtP3))
print('error rate:',1-accuracy_score(Y_test,dtP3))
print(classification_report(Y_test,dtP3))
kf = KFold(n_splits=10, shuffle=True)
mlp_model = MLPClassifier(hidden_layer_sizes=(10,), max_iter=100, random_state=42)
score = cross_val_score(mlp_model, result, y, cv= kf, scoring="accuracy")
print("%0.4f accuracy with a standard deviation of %0.4f" % (score.mean(), score.std()))
cf_matrix=confusion_matrix(Y_test,dtP3)
cf_matrix

Training time: 8.117755651473999s
accuracy score 0.9686711801407142
error rate: 0.03132881985928582
              precision    recall  f1-score   support

       DrDoS       0.97      0.88      0.92      5873
        LDAP       0.96      0.99      0.97      5905
       MSSQL       0.95      0.97      0.96      5848
     NetBIOS       0.95      0.96      0.96      5900
      Normal       1.00      1.00      1.00      5855
     Portmap       0.95      0.94      0.95      5803
         Syn       1.00      0.99      1.00      5871
        TFTP       1.00      1.00      1.00      5870
         UDP       0.94      0.98      0.96      5806

    accuracy                           0.97     52731
   macro avg       0.97      0.97      0.97     52731
weighted avg       0.97      0.97      0.97     52731

0.9706 accuracy with a standard deviation of 0.0008


array([[5167,  206,  168,   21,   13,   17,    2,    1,  278],
       [  11, 5857,   37,    0,    0,    0,    0,    0,    0],
       [  82,   47, 5679,    7,    1,   10,    0,    3,   19],
       [   1,    0,    1, 5684,    0,  203,    0,    0,   11],
       [   7,    0,    3,    1, 5833,    4,    1,    4,    2],
       [   9,    0,   28,  271,    0, 5482,    4,    0,    9],
       [   1,    0,    1,    1,    3,   12, 5835,    1,   17],
       [   1,    0,    6,    0,    1,    0,    0, 5855,    7],
       [  63,    0,   32,    2,    6,   14,    0,    2, 5687]])

In [ ]:
array1=[[5445,  136,  183,   13,    6,   12,    1,    0,   77],
       [  11, 5869,   25,    0,    0,    0,    0,    0,    0],
       [  36,   23, 5758,    0,    1,   13,    0,    0,   17],
       [   3,    0,    0, 5729,    0,  167,    0,    0,    1],
       [   7,    0,    0,    1, 5840,    1,    1,    4,    1],
       [   3,    0,    6,  228,    0, 5559,    3,    0,    4],
       [   0,    0,    1,    0,    0,    8, 5852,    0,   10],
       [   3,    0,    1,    0,    1,    0,    2, 5856,    7],
       [ 141,    0,   30,    0,    1,   10,    5,    2, 5617]]

array2=[[5157,  214,  168,   20,   12,   17,    2,    1,  282],
       [   4, 5865,   36,    0,    0,    0,    0,    0,    0],
       [  82,   49, 5681,    3,    1,   12,    0,    3,   17],
       [   1,    0,    0, 5697,    0,  202,    0,    0,    0],
       [   8,    0,    3,    1, 5831,    4,    1,    4,    3],
       [   9,    0,   34,  270,    0, 5478,    4,    0,    8],
       [   1,    0,    1,    0,    2,   13, 5837,    0,   17],
       [   0,    0,    6,    0,    1,    0,    3, 5853,    7],
       [  63,    0,   31,    2,    1,   15,    0,    2, 5692]]

array3=[[5432,  144,  185,   15,    5,   11,    1,    1,   79],
       [  12, 5863,   29,    0,    0,    0,    0,    0,    1],
       [  73,   21, 5703,    0,    1,   12,    2,    0,   36],
       [   4,    0,    0, 5718,    0,  177,    0,    0,    1],
       [  10,    0,    0,    2, 5837,    1,    2,    2,    1],
       [   8,    0,   12,  271,    1, 5494,   11,    1,    5],
       [   1,    0,    0,    0,    2,    3, 5855,    2,    8],
       [   1,    0,    0,    0,    2,    0,    2, 5862,    3],
       [ 140,    0,   17,    1,    2,   13,   15,    5, 5613]]

array4=[[5407,  158,  221,   19,   11,   15,    3,    0,   39],
       [   4, 5889,   12,    0,    0,    0,    0,    0,    0],
       [  30,   42, 5740,    0,    1,   17,    0,    0,   18],
       [   1,    0,    0, 5722,    0,  177,    0,    0,    0],
       [   5,    0,    0,    1, 5841,    4,    1,    3,    0],
       [   3,    0,    6,  267,    0, 5511,    3,    0,   13],
       [   1,    0,    1,    0,    1,   10, 5842,    1,   15],
       [   0,    0,    1,    0,    1,    0,    2, 5859,    7],
       [ 194,    0,   29,    1,    2,    8,    0,    1, 5571]]
array5=[[5422,  157,  260,   14,    8,    7,    0,    0,    5],
       [  42, 5700,  120,    0,   42,    0,    0,    0,    1],
       [  11,   30, 5806,    0,    0,    0,    0,    1,    0],
       [  12,    0,    3, 5308,  349,  198,    0,    0,   30],
       [  14,    0,    4,    0, 5827,    4,    2,    0,    4],
       [  29,    0,   47,  238,   46, 5406,    2,    0,   35],
       [   3,    1,    1,    0,   50,   13, 5784,    0,   19],
       [   3,    0,    0,    0,   13,    0,    0, 5844,   10],
       [ 300,    0,   68,    0,    1,    7,    0,    0, 5430]]
array6=[[5167,  206,  168,   21,   13,   17,    2,    1,  278],
       [  11, 5857,   37,    0,    0,    0,    0,    0,    0],
       [  82,   47, 5679,    7,    1,   10,    0,    3,   19],
       [   1,    0,    1, 5684,    0,  203,    0,    0,   11],
       [   7,    0,    3,    1, 5833,    4,    1,    4,    2],
       [   9,    0,   28,  271,    0, 5482,    4,    0,    9],
       [   1,    0,    1,    1,    3,   12, 5835,    1,   17],
       [   1,    0,    6,    0,    1,    0,    0, 5855,    7],
       [  63,    0,   32,    2,    6,   14,    0,    2, 5687]]
# Create a figure with 8 subplots
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(13,8),dpi=150)

# Create a list of confusion matrices and corresponding titles
cms =    [array1, array2, array3, array4,array5,array6]
titles = ['RF'  ,'LR',   'KNC',   'GBC','DT','MLP' ]

# Plot each confusion matrix as a heatmap on a subplot
for ax, cm, title in zip(axes.flat, cms, titles):
    sns.heatmap(cm, annot=True, cmap='Blues', ax=ax,fmt='g')
    ax.set_title(title)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
# axes.flat[-1].axis('off')
# Adjust spacing between subplots
plt.subplots_adjust(hspace=0.4, wspace=0.4)
fig.tight_layout()
# Display the figure
# plt.savefig('froncm.pdf')
plt.show()

# data leakage validation

In [8]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_predict, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y = df['Label']
x = df.drop(['Label'], axis=1)

X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.2, random_state=0, shuffle=True)


knc = KNeighborsClassifier(n_neighbors=3)
gb = GradientBoostingClassifier(n_estimators=4, learning_rate=0.1, max_depth=4, random_state=0)
kf_train = KFold(n_splits=5, shuffle=True, random_state=0)
knc_oof_train = cross_val_predict(knc, X_train, Y_train, cv=kf_train, method='predict_proba')
gb_oof_train = cross_val_predict(gb, X_train, Y_train, cv=kf_train, method='predict_proba')

meta_X_train = np.hstack((gb_oof_train, knc_oof_train))
knc.fit(X_train, Y_train)
gb.fit(X_train, Y_train)

knc_test_pred = knc.predict_proba(X_test)
gb_test_pred = gb.predict_proba(X_test)

meta_X_test = np.hstack((gb_test_pred, knc_test_pred))
clf = RandomForestClassifier(n_estimators=20, max_depth=20, random_state=0)
start = time.time()
clf.fit(meta_X_train, Y_train)
stop = time.time()
dtP3 = clf.predict(meta_X_test)
print(f"Training time: {stop - start}s")
print('Leak-Free Accuracy Score:', accuracy_score(Y_test, dtP3))
print('Error rate:', 1 - accuracy_score(Y_test, dtP3))
print(classification_report(Y_test, dtP3))
print("Confusion Matrix:\n", confusion_matrix(Y_test, dtP3))

Training time: 1.214583396911621s
Leak-Free Accuracy Score: 0.9465082584758041
Error rate: 0.05349174152419589
              precision    recall  f1-score   support

       DrDoS       0.95      0.91      0.93      5867
        LDAP       0.20      0.03      0.06        58
       MSSQL       0.55      0.45      0.50       348
     NetBIOS       0.17      0.05      0.08        19
      Normal       0.99      1.00      0.99      3998
     Portmap       0.15      0.15      0.15        27
         Syn       1.00      0.99      0.99      1962
        TFTP       1.00      1.00      1.00      3907
         UDP       0.72      0.95      0.82      1069

    accuracy                           0.95     17255
   macro avg       0.63      0.61      0.61     17255
weighted avg       0.95      0.95      0.95     17255

Confusion Matrix:
 [[5343    7  111    4   16   10    0    0  376]
 [  54    2    2    0    0    0    0    0    0]
 [ 184    1  157    0    0    0    0    0    6]
 [   6    0    0    1